# Experiment 2 — multi-hop на gold-пассажах

В `test_HRM.ipynb` мы увидели, что модель умеет брать один факт из контекста.

Здесь хочу проверить следующий шаг перед ретривером и HotpotQA (или другим бенчмарком):

1. Умеет ли модель **скомбинировать два пассажа** (multi-hop)?
2. Не ломается ли ответ, если в контекст подмешан **шум**?

Ретривера нет — все пассажи «золотые».

## 0. Загрузка модели

Тот же формат PrefixLM, что и в первой тетрадке.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("sapientinc/HRM-Text-1B")
model = AutoModelForCausalLM.from_pretrained(
    "sapientinc/HRM-Text-1B",
    dtype=torch.float32,
    attn_implementation="sdpa",
).to("cpu").eval()

CONDITION = "<|quad_end|><|object_ref_end|>"
print("ok")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/326 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.73M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.37GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

ok


## 1. Одна простая функция `ask`

Передаём список пассажей + вопрос → печатаем ответ модели.

In [ ]:
def ask(passages, question, max_new_tokens=40):
    """passages: list[str]. Пустой список = вопрос без контекста."""
    if passages:
        ctx = "\n".join(f"[{i}] {p}" for i, p in enumerate(passages, 1))
        body = f"Context:\n{ctx}\n\nQuestion: {question}\nAnswer:"
    else:
        body = f"Question: {question}\nAnswer:"

    prompt = f"<|im_start|>{CONDITION}{body}<|im_end|>"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    inputs["token_type_ids"] = torch.ones_like(inputs["input_ids"])

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    text = tokenizer.decode(out[0], skip_special_tokens=False)
    # печатаем только то, что модель сгенерировала после <|im_end|>
    answer = text.split("<|im_end|>", 1)[-1].strip()
    print(answer)
    return answer

## 2. Multi-hop: нужны оба пассажа

Ответ **не лежит** ни в одном пассаже по отдельности — надо связать их.

**Ожидание:** правильный ответ только когда есть оба пассажа.

In [ ]:
P1 = "Alice works at Acme Corp in Lyon."
P2 = "Acme Corp was founded in 1994."
Q = "In what year was Alice's employer founded?"

In [ ]:
print("=== оба пассажа (ожидаем 1994) ===")
ask([P1, P2], Q)

print("\n=== только P1 (факта про год нет) ===")
ask([P1], Q)

print("\n=== только P2 (не сказано, что Acme = работодатель Alice) ===")
ask([P2], Q)

print("\n=== без контекста ===")
ask([], Q)

=== оба пассажа (ожидаем 1994) ===


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:2613: UserWarning: You are calling .generate() with the `input_ids` being on a device type different than your model's device. `input_ids` is on cuda, whereas the model is on cpu. You may experience unexpected behaviors or slower generation. Please make sure that you have put `input_ids` to the correct device by calling for example input_ids = input_ids.to('cpu') before running `.generate()`.
  warnings.warn(


Alice's employer, Acme Corp, was founded in 1994. 

**Answer:** 1994<|box_end|>

=== только P1 (факта про год нет) ===
Alice's employer, Acme Corp., was founded in 1995. 

\boxed{1995}<|box_end|>

=== только P2 (не сказано, что Acme = работодатель Alice) ===
The problem states that Acme Corp was founded in 1994. Since the question asks for the year Alice's employer was founded, and given that Ac

=== без контекста ===
The question asks for the year Alice's employer was founded. However, the reference to "Alice's employer" is ambiguous without additional context. If we assume t


'The question asks for the year Alice\'s employer was founded. However, the reference to "Alice\'s employer" is ambiguous without additional context. If we assume t'

### Анализ (Alice / Acme)

| Условие | Ответ модели | Вердикт |
|---------|--------------|---------|
| оба пассажа | `1994` | правильно связала Alice → Acme → 1994 |
| только P1 | `1995` | уверенная галлюцинация (года в тексте нет) |
| только P2 | обрывок про 1994, без явной связи с Alice | частично «подтягивает» год из Acme |
| без контекста | честно пишет, что контекста не хватает | хороший отказ |

**Вывод:** multi-hop работает, когда оба факта на месте. Без полного контекста модель чаще галлюцинирует, чем отказывается (особенно на P1).


## 3. Ещё два multi-hop примера

Те же правила: оба пассажа vs один vs ничего.

In [ ]:
examples = [
    {
        "p1": "The Kelso River flows through the city of Norvenna.",
        "p2": "Norvenna has a population of 340,000 people.",
        "q": "What is the population of the city that the Kelso River flows through?",
        "gold": "340,000",
    },
    {
        "p1": "Dr. Elena Foss discovered the mineral Rathenite.",
        "p2": "Rathenite was discovered in 1887.",
        "q": "In what year did Dr. Elena Foss make her mineral discovery?",
        "gold": "1887",
    },
]

for ex in examples:
    print("=" * 60)
    print("Q:", ex["q"])
    print("gold:", ex["gold"])
    print()

    print("--- оба пассажа ---")
    ask([ex["p1"], ex["p2"]], ex["q"])

    print("\n--- только p1 ---")
    ask([ex["p1"]], ex["q"])

    print("\n--- только p2 ---")
    ask([ex["p2"]], ex["q"])

    print("\n--- без контекста ---")
    ask([], ex["q"])
    print()

Q: What is the population of the city that the Kelso River flows through?
gold: 340,000

--- оба пассажа ---
The population of the city that the Kelso River flows through is \boxed{340000}.<|box_end|>

--- только p1 ---
The population of the city that the Kelso River flows through is \boxed{100,000}.<|box_end|>

--- только p2 ---
The Kelso River flows through the city of Kelso, which is located in the state of Nigeria. The population of Kelso is a well-known figure,

--- без контекста ---
The Kelso River flows through the city of **Kololo**. To determine its population, we can follow these steps:

1. **Identify the city**: The

Q: In what year did Dr. Elena Foss make her mineral discovery?
gold: 1887

--- оба пассажа ---
Dr. Elena Foss discovered the mineral Rathenite in 1887. 

**Answer:** 1887<|box_end|>

--- только p1 ---
Dr. Elena Foss discovered the mineral Rathenite in **1922**. This discovery was made by her team at the University of California, Berkeley, and

--- только p2 ---


### Анализ (ещё два примера)

**Kelso / Norvenna (gold = 340,000)**
- оба пассажа → `340000`
- только p1 → `100,000` галлюцинация
- только p2 / без контекста → выдумывает город и население

**Elena Foss / Rathenite (gold = 1887)**
- оба пассажа → `1887`
- только p1 → `1922` галлюцинация
- только p2 → осторожный текст без жёсткого ответа
- без контекста → `1980` + вымышленный минерал

**Вывод:** на 3/3 вымышленных multi-hop парах с двумя gold-пассажами ответ верный. Паттерн стабильный: полный контекст → grounding; неполный → confabulation.


## 4. Шум в контексте (лёгкий H2-check)

К правильным двум пассажам добавляем **irrelevant** предложение.

**Ожидание:** ответ всё ещё `1994` (модель не уходит в шум).

In [ ]:
NOISE = "The Eiffel Tower was completed in 1889 and is located in Paris, France."

print("=== чистый multi-hop ===")
ask([P1, P2], Q)

print("\n=== multi-hop + шум в конце ===")
ask([P1, P2, NOISE], Q)

print("\n=== шум первым (сложнее) ===")
ask([NOISE, P1, P2], Q)

=== чистый multi-hop ===
Alice's employer, Acme Corp, was founded in 1994. 

**Answer:** 1994<|box_end|>

=== multi-hop + шум в конце ===
Alice's employer is Acme Corp, which was founded in 1994. The Eiffel Tower, completed in 1889, is unrelated to this question. 

**Answer

=== шум первым (сложнее) ===
Alice's employer is Acme Corp, which was founded in 1994. 

**Answer:** 1994<|box_end|>


"Alice's employer is Acme Corp, which was founded in 1994. \n\n**Answer:** 1994<|box_end|>"

### Анализ (шум)

| Условие | Ответ | Вердикт |
|---------|-------|---------|
| чистый multi-hop | `1994` | + |
| шум в конце (Eiffel 1889) | всё ещё `1994`, шум явно помечен как unrelated | + |
| шум первым | `1994` | + |

**Вывод:** лёгкий irrelevant noise не сбивает ответ. Модель даже иногда вербализует, что Eiffel Tower к вопросу не относится — хороший знак для H2, но это пока toy-шум, не полный noise-benchmark.



## 5. Итог

| Проверка | Результат |
|----------|-----------|
| Multi-hop с двумя пассажами | работает (3/3: Acme, Norvenna, Foss) |
| С одним пассажем не «угадывает» правильно | нет — часто галлюцинирует (1995, 100k, 1922…) |
| Шум не сбивает ответ | да (на этом toy-примере) |

### Главные выводы

1. **Reader готов к multi-hop на gold context.** PrefixLM + два пассажа достаточно, чтобы HRM-Text-1B связал факты — зеленый свет к HotpotQA *supporting paragraphs* без своего ретривера.
2. **Без полного контекста модель выдумывает**, а не говорит «не знаю» (кроме пустого контекста на Acme). Для RAG это нормальная мотивация: retrieval критичен; «дырявый» top-k даст правдоподобный мусор.
3. **Устойчивость к лёгкому шуму есть**, но это ещё не H2 — нужен контролируемый шум на реальном бенчмарке.
